In [1]:
"""
Phase 0 MVP: Phonological Feature Probing Pipeline
Complete implementation from labels to evaluation
"""

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pickle
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from collections import defaultdict
import pandas as pd

# ============================================================================
# STEP 1: CREATE PHONOLOGICAL FEATURE LABELS
# ============================================================================

# Phoneme to phonological features mapping (IPA-based)
# Features: voicing, manner, place, nasality
PHONEME_FEATURES = {
    # Consonants
    'p': {'voicing': 'voiceless', 'manner': 'stop', 'place': 'bilabial', 'nasal': False},
    'b': {'voicing': 'voiced', 'manner': 'stop', 'place': 'bilabial', 'nasal': False},
    't': {'voicing': 'voiceless', 'manner': 'stop', 'place': 'alveolar', 'nasal': False},
    'd': {'voicing': 'voiced', 'manner': 'stop', 'place': 'alveolar', 'nasal': False},
    'k': {'voicing': 'voiceless', 'manner': 'stop', 'place': 'velar', 'nasal': False},
    'g': {'voicing': 'voiced', 'manner': 'stop', 'place': 'velar', 'nasal': False},
    'f': {'voicing': 'voiceless', 'manner': 'fricative', 'place': 'labiodental', 'nasal': False},
    'v': {'voicing': 'voiced', 'manner': 'fricative', 'place': 'labiodental', 'nasal': False},
    's': {'voicing': 'voiceless', 'manner': 'fricative', 'place': 'alveolar', 'nasal': False},
    'z': {'voicing': 'voiced', 'manner': 'fricative', 'place': 'alveolar', 'nasal': False},
    'ʃ': {'voicing': 'voiceless', 'manner': 'fricative', 'place': 'postalveolar', 'nasal': False},
    'ʒ': {'voicing': 'voiced', 'manner': 'fricative', 'place': 'postalveolar', 'nasal': False},
    'h': {'voicing': 'voiceless', 'manner': 'fricative', 'place': 'glottal', 'nasal': False},
    'm': {'voicing': 'voiced', 'manner': 'nasal', 'place': 'bilabial', 'nasal': True},
    'n': {'voicing': 'voiced', 'manner': 'nasal', 'place': 'alveolar', 'nasal': True},
    'ŋ': {'voicing': 'voiced', 'manner': 'nasal', 'place': 'velar', 'nasal': True},
    'l': {'voicing': 'voiced', 'manner': 'lateral', 'place': 'alveolar', 'nasal': False},
    'r': {'voicing': 'voiced', 'manner': 'approximant', 'place': 'alveolar', 'nasal': False},
    'w': {'voicing': 'voiced', 'manner': 'approximant', 'place': 'bilabial', 'nasal': False},
    'j': {'voicing': 'voiced', 'manner': 'approximant', 'place': 'palatal', 'nasal': False},
    # Vowels (simplified)
    'i': {'voicing': 'voiced', 'manner': 'vowel', 'place': 'front', 'nasal': False},
    'e': {'voicing': 'voiced', 'manner': 'vowel', 'place': 'front', 'nasal': False},
    'a': {'voicing': 'voiced', 'manner': 'vowel', 'place': 'central', 'nasal': False},
    'o': {'voicing': 'voiced', 'manner': 'vowel', 'place': 'back', 'nasal': False},
    'u': {'voicing': 'voiced', 'manner': 'vowel', 'place': 'back', 'nasal': False},
    'ə': {'voicing': 'voiced', 'manner': 'vowel', 'place': 'central', 'nasal': False},
}

def extract_phonological_features(text):
    """
    Extract phonological features from text
    Simple approach: use lowercase text as approximation to phonemes
    For MVP - in production use phonemizer library
    """
    text_lower = text.lower()
    
    # Collect all features from the text
    features_found = defaultdict(list)
    
    for char in text_lower:
        if char in PHONEME_FEATURES:
            phoneme_feats = PHONEME_FEATURES[char]
            for feat_name, feat_value in phoneme_feats.items():
                features_found[feat_name].append(feat_value)
    
    # Return most common feature for each type
    aggregated_features = {}
    for feat_name, feat_values in features_found.items():
        if feat_values:
            # For boolean, use any
            if isinstance(feat_values[0], bool):
                aggregated_features[feat_name] = any(feat_values)
            else:
                # For categorical, use most common
                aggregated_features[feat_name] = max(set(feat_values), key=feat_values.count)
    
    return aggregated_features

# ============================================================================
# STEP 2: PREPARE DATASET WITH ALIGNED REPRESENTATIONS
# ============================================================================

def prepare_probing_dataset(features_file, layer_idx=6):
    """
    Prepare dataset for probing
    Uses utterance-level: average hidden states over time
    
    Args:
        features_file: Path to extracted features pickle
        layer_idx: Which layer to use (0-12)
    
    Returns:
        X: numpy array of representations
        y: dict of labels for each phonological feature
        metadata: sample information
    """
    print(f"Loading features from {features_file}")
    with open(features_file, 'rb') as f:
        features = pickle.load(f)
    
    X = []
    y_voicing = []
    y_manner = []
    y_place = []
    y_nasal = []
    metadata = []
    
    for sample in features:
        # Get hidden states for this layer
        hidden_state = sample['hidden_states'][layer_idx]  # Shape: [1, time, 768]
        
        # Average over time dimension (utterance-level representation)
        averaged = hidden_state.mean(axis=1).squeeze()  # Shape: [768]
        
        # Extract phonological features from transcription
        phon_features = extract_phonological_features(sample['transcription'])
        
        # Only keep samples where we found features
        if phon_features:
            X.append(averaged)
            
            # Encode labels
            y_voicing.append(phon_features.get('voicing', 'unknown'))
            y_manner.append(phon_features.get('manner', 'unknown'))
            y_place.append(phon_features.get('place', 'unknown'))
            y_nasal.append(1 if phon_features.get('nasal', False) else 0)
            
            metadata.append({
                'id': sample['id'],
                'language': sample['language'],
                'transcription': sample['transcription']
            })
    
    X = np.array(X)
    
    # Create label dictionaries
    y = {
        'voicing': np.array(y_voicing),
        'manner': np.array(y_manner),
        'place': np.array(y_place),
        'nasal': np.array(y_nasal)
    }
    
    print(f"Prepared {len(X)} samples")
    print(f"Representation shape: {X.shape}")
    print(f"Label distributions:")
    for feat_name, labels in y.items():
        unique, counts = np.unique(labels, return_counts=True)
        print(f"  {feat_name}: {dict(zip(unique, counts))}")
    
    return X, y, metadata

# ============================================================================
# STEP 3: TRAIN PROBING CLASSIFIERS
# ============================================================================

def train_probe(X_train, y_train, feature_name):
    """
    Train a simple linear probe (logistic regression)
    
    Args:
        X_train: Training representations
        y_train: Training labels
        feature_name: Name of the feature being probed
    
    Returns:
        Trained classifier
    """
    print(f"\nTraining probe for: {feature_name}")
    
    # Use logistic regression as probe
    probe = LogisticRegression(
        max_iter=1000,
        random_state=42,
        multi_class='multinomial'
    )
    
    probe.fit(X_train, y_train)
    
    # Training accuracy
    train_acc = probe.score(X_train, y_train)
    print(f"  Training accuracy: {train_acc:.3f}")
    
    return probe

def train_all_probes(X_train, y_train_dict):
    """
    Train probes for all phonological features
    
    Returns:
        Dictionary of trained probes
    """
    probes = {}
    
    for feature_name, y_train in y_train_dict.items():
        probe = train_probe(X_train, y_train, feature_name)
        probes[feature_name] = probe
    
    return probes

# ============================================================================
# STEP 4: EVALUATE PROBES
# ============================================================================

def evaluate_probe(probe, X_test, y_test, feature_name, test_lang):
    """
    Evaluate a probe on test data
    
    Returns:
        Dictionary of metrics
    """
    y_pred = probe.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    
    print(f"\n{feature_name} on {test_lang}:")
    print(f"  Accuracy: {accuracy:.3f}")
    
    # Detailed report
    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    
    return {
        'feature': feature_name,
        'language': test_lang,
        'accuracy': accuracy,
        'report': report
    }

def evaluate_all_probes(probes, X_test, y_test_dict, test_lang):
    """
    Evaluate all probes on test data
    
    Returns:
        List of evaluation results
    """
    results = []
    
    for feature_name, probe in probes.items():
        y_test = y_test_dict[feature_name]
        result = evaluate_probe(probe, X_test, y_test, feature_name, test_lang)
        results.append(result)
    
    return results

# ============================================================================
# CROSS-LINGUAL EVALUATION
# ============================================================================

def cross_lingual_evaluation(train_lang, test_langs, layer_idx=6, features_dir="./extracted_features"):
    """
    Main cross-lingual evaluation pipeline
    Train on one language, test on others
    
    Args:
        train_lang: Language to train on (e.g., 'en_us')
        test_langs: List of languages to test on (e.g., ['de_de', 'es_419'])
        layer_idx: Which layer to probe (0-12)
        features_dir: Directory with extracted features
    """
    print("="*60)
    print(f"Cross-Lingual Probing Evaluation")
    print(f"Layer: {layer_idx}")
    print(f"Train language: {train_lang}")
    print(f"Test languages: {test_langs}")
    print("="*60)
    
    # 1. Prepare training data
    train_file = f"{features_dir}/{train_lang}_features.pkl"
    X_train, y_train, meta_train = prepare_probing_dataset(train_file, layer_idx)
    
    # 2. Train probes
    print(f"\n{'='*60}")
    print("Training Probes")
    print("="*60)
    probes = train_all_probes(X_train, y_train)
    
    # 3. Evaluate on test languages
    all_results = []
    
    for test_lang in test_langs:
        print(f"\n{'='*60}")
        print(f"Evaluating on: {test_lang}")
        print("="*60)
        
        test_file = f"{features_dir}/{test_lang}_features.pkl"
        X_test, y_test, meta_test = prepare_probing_dataset(test_file, layer_idx)
        
        results = evaluate_all_probes(probes, X_test, y_test, test_lang)
        all_results.extend(results)
    
    # 4. Summary table
    print(f"\n{'='*60}")
    print("SUMMARY")
    print("="*60)
    
    df_results = pd.DataFrame([
        {
            'Feature': r['feature'],
            'Language': r['language'],
            'Accuracy': f"{r['accuracy']:.3f}"
        }
        for r in all_results
    ])
    
    print(df_results.to_string(index=False))
    
    return all_results, probes

# ============================================================================
# LAYER COMPARISON
# ============================================================================

def compare_layers(lang, features_dir="./extracted_features", layers=[0, 3, 6, 9, 12]):
    """
    Compare probing performance across different layers
    Train and test on same language, but different layers
    """
    print("="*60)
    print(f"Layer Comparison for {lang}")
    print(f"Testing layers: {layers}")
    print("="*60)
    
    layer_results = []
    
    for layer_idx in layers:
        print(f"\n{'='*60}")
        print(f"Layer {layer_idx}")
        print("="*60)
        
        # Prepare data
        features_file = f"{features_dir}/{lang}_features.pkl"
        X, y, meta = prepare_probing_dataset(features_file, layer_idx)
        
        # Simple train/test split (80/20)
        split_idx = int(0.8 * len(X))
        X_train, X_test = X[:split_idx], X[split_idx:]
        y_train = {k: v[:split_idx] for k, v in y.items()}
        y_test = {k: v[split_idx:] for k, v in y.items()}
        
        # Train probes
        probes = train_all_probes(X_train, y_train)
        
        # Evaluate
        results = evaluate_all_probes(probes, X_test, y_test, f"layer_{layer_idx}")
        
        for r in results:
            r['layer'] = layer_idx
            layer_results.append(r)
    
    # Summary
    print(f"\n{'='*60}")
    print("LAYER COMPARISON SUMMARY")
    print("="*60)
    
    df = pd.DataFrame([
        {
            'Layer': r['layer'],
            'Feature': r['feature'],
            'Accuracy': f"{r['accuracy']:.3f}"
        }
        for r in layer_results
    ])
    
    # Pivot table for easy viewing
    pivot = df.pivot(index='Feature', columns='Layer', values='Accuracy')
    print(pivot.to_string())
    
    return layer_results



In [2]:
print("\n" + "="*60)
print("PHASE 0 MVP: PHONOLOGICAL PROBING PIPELINE")
print("="*60)

# ========================================================================
# EXPERIMENT 1: Cross-Lingual Transfer
# ========================================================================

print("\n\n" + "="*60)
print("EXPERIMENT 1: Cross-Lingual Transfer")
print("="*60)

# Train on English, test on German and Spanish
cross_lingual_results, probes = cross_lingual_evaluation(
    train_lang="en_us",
    test_langs=["de_de", "es_419"],
    layer_idx=6  # Middle layer
)

# ========================================================================
# EXPERIMENT 2: Layer Comparison
# ========================================================================

print("\n\n" + "="*60)
print("EXPERIMENT 2: Layer Comparison")
print("="*60)

# Compare different layers on English
layer_results = compare_layers(
    lang="en_us",
    layers=[0, 3, 6, 9, 12]
)

# ========================================================================
# Save results
# ========================================================================

results_dict = {
    'cross_lingual': cross_lingual_results,
    'layer_comparison': layer_results
}

with open("./probing_results.pkl", 'wb') as f:
    pickle.dump(results_dict, f)

print("\n" + "="*60)
print("Results saved to: probing_results.pkl")
print("="*60)


PHASE 0 MVP: PHONOLOGICAL PROBING PIPELINE


EXPERIMENT 1: Cross-Lingual Transfer
Cross-Lingual Probing Evaluation
Layer: 6
Train language: en_us
Test languages: ['de_de', 'es_419']
Loading features from ./extracted_features/en_us_features.pkl


FileNotFoundError: [Errno 2] No such file or directory: './extracted_features/en_us_features.pkl'